# Aspect-Based Sentiment Analysis for Restaurant Reviews
### Academic NLP Project: SemEval-2014 Task 4 Benchmark

This notebook demonstrates the end-to-end NLP pipeline for Aspect-Based Sentiment Analysis (ABSA):
1. **Dataset Exploration**: Inspecting the SemEval-2014 Task 4 Restaurant dataset annotations.
2. **POS Tagging & Aspect Extraction**: Detecting restaurant entities using POS chunking & domain lexicon.
3. **Clause Boundary Splitting**: Extracting isolated local context to prevent sentiment bleeding.
4. **TF-IDF Feature Representation**: Extracting unigrams and bigrams.
5. **Model Evaluation**: Comparing Logistic Regression (Balanced) against Naive Bayes (Baseline).
6. **Inference**: Testing mixed-sentiment reviews (`"The food was amazing but the service was extremely slow."`).

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_dataset
from src.pipeline import ABSAPipeline
from src.classifier import ABSAClassifier

## 1. Load SemEval-2014 Dataset

In [ ]:
train_df, test_df = load_dataset('../data')
print(f'Training aspects: {len(train_df)}')
print(f'Testing aspects: {len(test_df)}')
train_df.head(5)

## 2. Polarity Distribution & Class Imbalance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
train_df['polarity'].value_counts().plot(kind='bar', color=['#10B981', '#EF4444', '#6B7280'], ax=axes[0])
axes[0].set_title('Training Polarity Distribution')
axes[0].set_ylabel('Count')

test_df['polarity'].value_counts().plot(kind='bar', color=['#10B981', '#EF4444', '#6B7280'], ax=axes[1])
axes[1].set_title('Testing Polarity Distribution')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()

## 3. Most Frequently Annotated Restaurant Aspects

In [ ]:
top_aspects = train_df['aspect_term'].str.lower().value_counts().head(12)
plt.figure(figsize=(9, 4))
top_aspects.plot(kind='barh', color='#3B82F6').invert_yaxis()
plt.title('Top 12 Restaurant Aspects in Training Corpus')
plt.xlabel('Frequency')
plt.show()

## 4. Run ABSA Pipeline on Mixed-Sentiment Review

In [ ]:
pipeline = ABSAPipeline(models_dir='../models')
review = 'The food was amazing but the service was extremely slow.'
result = pipeline.analyze_review(review)

print('Original Review:', result['review'])
print('Overall Sentiment:', result['overall_sentiment'])
print('\nAspect Results:')
for a in result['aspect_results']:
    print(f"  - Aspect: {a['aspect']:<10} -> {a['sentiment'].upper()} (Confidence: {a['confidence']*100:.1f}%)")
    print(f"    Context: '{a['context_used']}'")